In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.fact_orders AS
WITH s1 AS (
  SELECT * FROM workspace.silver.sub_order_timeline WHERE sub_order_seq = 1
),
items AS (
  SELECT order_id,
         COUNT(*)                                                         AS n_lines,
         SUM(qty_ordered)                                                 AS units_ordered,
         SUM(qty_fulfilled)                                               AS units_fulfilled,
         SUM(ordered_value)                                               AS ordered_value,
         SUM(CASE WHEN was_unavailable THEN 1 ELSE 0 END)                 AS lines_unavailable,
         SUM(CASE WHEN item_status = 'substituted'     THEN 1 ELSE 0 END) AS lines_substituted,
         SUM(CASE WHEN item_status = 'removed'         THEN 1 ELSE 0 END) AS lines_removed,
         SUM(CASE WHEN item_status = 'fulfilled_split' THEN 1 ELSE 0 END) AS lines_split,
         SUM(CASE WHEN item_status = 'removed' THEN ordered_value ELSE 0 END) AS removed_value
  FROM workspace.silver.order_items
  GROUP BY order_id
)
SELECT o.order_id, o.customer_id, o.store_id, o.placed_ts, o.placed_date, o.placed_hour,
       c.is_rainy, c.is_weekend, o.promised_minutes, o.order_status, o.cancel_reason,
       o.is_split, o.n_sub_orders, o.delivered_ts,
       ROUND((unix_timestamp(o.delivered_ts) - unix_timestamp(o.placed_ts)) / 60.0D, 2) AS delivery_minutes,
       CASE WHEN o.order_status = 'delivered'
            THEN (unix_timestamp(o.delivered_ts) - unix_timestamp(o.placed_ts)) / 60.0D <= o.promised_minutes
       END AS is_on_time,
       -- stage durations of the FIRST shipment, in minutes
       ROUND((unix_timestamp(o.accepted_ts) - unix_timestamp(o.placed_ts)) / 60.0D, 2)                     AS accept_min,
       ROUND((unix_timestamp(s1.picking_started_ts) - unix_timestamp(o.accepted_ts)) / 60.0D, 2)           AS pick_wait_min,
       ROUND((unix_timestamp(s1.packing_completed_ts) - unix_timestamp(s1.picking_started_ts)) / 60.0D, 2) AS pick_pack_min,
       ROUND((unix_timestamp(s1.picked_up_ts) - unix_timestamp(s1.packing_completed_ts)) / 60.0D, 2)       AS rider_wait_min,
       ROUND((unix_timestamp(s1.delivered_ts) - unix_timestamp(s1.picked_up_ts)) / 60.0D, 2)               AS ride_min,
       i.n_lines, i.units_ordered, i.units_fulfilled, i.ordered_value,
       i.lines_unavailable, i.lines_substituted, i.lines_removed, i.lines_split, i.removed_value
FROM workspace.silver.orders o
LEFT JOIN s1 ON o.order_id = s1.order_id
LEFT JOIN items i ON o.order_id = i.order_id
LEFT JOIN workspace.bronze.daily_context c ON o.placed_date = c.date;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.store_hour_metrics AS
SELECT store_id, placed_date, placed_hour, is_rainy,
       COUNT(*)                                                    AS orders,
       SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
       SUM(CASE WHEN order_status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
       SUM(CASE WHEN is_on_time THEN 1 ELSE 0 END)                 AS on_time_orders,
       ROUND(AVG(delivery_minutes), 2)                             AS avg_delivery_minutes,
       ROUND(AVG(rider_wait_min), 2)                               AS avg_rider_wait_min,
       ROUND(AVG(ride_min), 2)                                     AS avg_ride_min
FROM workspace.gold.fact_orders
GROUP BY store_id, placed_date, placed_hour, is_rainy;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.sku_availability AS
WITH inv AS (
  SELECT store_id, sku_id, CAST(snapshot_ts AS DATE) AS date,
         COUNT(*) AS hours_open,
         SUM(CASE WHEN on_hand_qty = 0 THEN 1 ELSE 0 END) AS hours_out_of_stock
  FROM workspace.bronze.inventory_snapshots
  GROUP BY store_id, sku_id, CAST(snapshot_ts AS DATE)
),
demand AS (
  SELECT o.store_id, i.sku_id, o.placed_date AS date,
         SUM(i.qty_ordered)                                                     AS units_ordered,
         SUM(CASE WHEN i.was_unavailable THEN 1 ELSE 0 END)                     AS unavailable_lines,
         SUM(CASE WHEN i.item_status = 'removed' THEN i.ordered_value ELSE 0 END) AS removed_value
  FROM workspace.silver.order_items i
  JOIN workspace.silver.orders o ON i.order_id = o.order_id
  GROUP BY o.store_id, i.sku_id, o.placed_date
)
SELECT inv.store_id, inv.sku_id, s.category, inv.date, inv.hours_open, inv.hours_out_of_stock,
       COALESCE(d.units_ordered, 0)     AS units_ordered,
       COALESCE(d.unavailable_lines, 0) AS unavailable_lines,
       COALESCE(d.removed_value, 0)     AS removed_value
FROM inv
JOIN workspace.bronze.skus s ON inv.sku_id = s.sku_id
LEFT JOIN demand d ON inv.store_id = d.store_id AND inv.sku_id = d.sku_id AND inv.date = d.date;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.fv_store_day AS
WITH w AS (
  SELECT store_id, date, SUM(qty_received) AS qty_received, SUM(qty_wasted) AS qty_wasted
  FROM workspace.bronze.wastage_log
  GROUP BY store_id, date
),
oos AS (
  SELECT store_id, date, SUM(hours_out_of_stock) AS oos_sku_hours, SUM(hours_open) AS sku_hours
  FROM workspace.gold.sku_availability
  WHERE category = 'Fruits & Vegetables'
  GROUP BY store_id, date
),
sold AS (
  SELECT o.store_id, o.placed_date AS date, SUM(i.qty_fulfilled) AS units_sold
  FROM workspace.silver.order_items i
  JOIN workspace.silver.orders o ON i.order_id = o.order_id
  JOIN workspace.bronze.skus s ON i.sku_id = s.sku_id
  WHERE s.category = 'Fruits & Vegetables' AND i.item_status = 'fulfilled'
  GROUP BY o.store_id, o.placed_date
)
SELECT w.store_id, w.date, w.qty_received, w.qty_wasted,
       oos.oos_sku_hours, oos.sku_hours,
       COALESCE(sold.units_sold, 0) AS units_sold
FROM w
JOIN oos ON w.store_id = oos.store_id AND w.date = oos.date
LEFT JOIN sold ON w.store_id = sold.store_id AND w.date = sold.date;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.customer_first_order AS
WITH ranked AS (
  SELECT customer_id, order_id, placed_ts, order_status, is_on_time, lines_removed,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY placed_ts) AS rn
  FROM workspace.gold.fact_orders
),
first_o AS (SELECT * FROM ranked WHERE rn = 1),
bounds  AS (SELECT MAX(placed_ts) AS max_ts FROM workspace.gold.fact_orders),
reorders AS (
  SELECT f.customer_id, COUNT(x.order_id) AS orders_next_7d
  FROM first_o f
  LEFT JOIN workspace.gold.fact_orders x
         ON x.customer_id = f.customer_id
        AND x.placed_ts > f.placed_ts
        AND x.placed_ts <= f.placed_ts + INTERVAL 7 DAYS
  GROUP BY f.customer_id
)
SELECT f.customer_id, f.order_id AS first_order_id, f.placed_ts AS first_placed_ts,
       (f.order_status = 'cancelled' OR f.is_on_time = FALSE OR f.lines_removed > 0) AS first_order_bad,
       r.orders_next_7d,
       (r.orders_next_7d > 0)                            AS reordered_7d,
       (f.placed_ts <= b.max_ts - INTERVAL 7 DAYS)       AS has_full_7d_window
FROM first_o f
JOIN reorders r ON f.customer_id = r.customer_id
CROSS JOIN bounds b;